# Install Dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libopenal1 libopenal-dev > /dev/null
!pip install -q vizdoom agilerl gymnasium imageio imageio-ffmpeg scipy numpy torch


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# Imports

In [ ]:
from __future__ import annotations

import os
import re
import subprocess

from pathlib import Path

import gymnasium
import imageio.v2 as imageio
import imageio_ffmpeg
import numpy as np
import torch
import vizdoom as vzd
from agilerl.algorithms.dqn_rainbow import RainbowDQN
from agilerl.components.replay_buffer import MultiStepReplayBuffer, PrioritizedReplayBuffer
from agilerl.training.train_off_policy import train_off_policy
from gymnasium.spaces import Box
from gymnasium.wrappers import FrameStackObservation, ResizeObservation, TransformObservation, TransformReward
from IPython.display import Video, display
from scipy.io import wavfile
from vizdoom import gymnasium_wrapper
from vizdoom.gymnasium_wrapper.base_gymnasium_env import VizdoomEnv
from datetime import datetime


# Experiment Configuration

In [ ]:
ENV_NAME = "VizdoomBasic-v1"
NUM_ENVS = 2
EVAL_EPISODES = 5
FRAME_SKIP = 4
STACK_SIZE = 4
OBS_SIZE = (84, 84)
MAX_BUTTONS_PRESSED = 1
use_doom_wad = True
use_custom_scenario = True



# Paths

In [ ]:
EXPERIMENT_NAME = "basic-dqn"
SCENARIO_CFG = "basic_outro.cfg"

try:
    from google.colab import drive

    drive.mount("/content/drive")
    repo_root = Path("/content/drive/MyDrive/rl-epsilon")
except ImportError:
    here = Path.cwd().resolve()
    repo_root = None
    for candidate in [here, *here.parents]:
        experiment = candidate / "vizdoom-experiments" / EXPERIMENT_NAME
        if (experiment / "scenario_files").is_dir():
            repo_root = candidate
            break
        if candidate.name == EXPERIMENT_NAME and (candidate / "scenario_files").is_dir():
            repo_root = candidate.parents[1]
            break
    if repo_root is None:
        raise FileNotFoundError(
            "Could not find the rl-epsilon repo. Run this notebook from inside the repo."
        )

experiment_dir = repo_root / "vizdoom-experiments" / EXPERIMENT_NAME
if not experiment_dir.is_dir():
    raise FileNotFoundError(f"Experiment directory not found: {experiment_dir}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = "run-" + timestamp
run_dir = experiment_dir / "training-runs" / run_name

checkpoint_dir = run_dir / "checkpoints"
recording_dir = run_dir / "recordings"
video_dir = run_dir / "video"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(recording_dir, exist_ok=True)
os.makedirs(video_dir, exist_ok=True)

DOOM_WAD = None
if use_doom_wad:
    DOOM_WAD = repo_root / "vizdoom-experiments" / "WADS" / "DOOM2.WAD"
    if not DOOM_WAD.is_file():
        raise FileNotFoundError(
            f"DOOM2.WAD not found at {DOOM_WAD}. Place your commercial IWAD there."
        )
    DOOM_WAD = str(DOOM_WAD)

custom_scenario_location = None
if use_custom_scenario:
    custom_scenario_location = experiment_dir / "scenario_files" / SCENARIO_CFG
    if not custom_scenario_location.is_file():
        raise FileNotFoundError(f"Scenario config not found: {custom_scenario_location}")
    custom_scenario_location = str(custom_scenario_location)

print(f"repo: {repo_root}")
print(f"experiment: {experiment_dir}")
print(f"checkpoints: {checkpoint_dir}")
print(f"recordings: {recording_dir}")
print(f"videos: {video_dir}")
print(f"scenario: {custom_scenario_location}")
print(f"wad: {DOOM_WAD}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Environment Creation

In [ ]:
def preprocess_env(env):
    h, w, _ = env.observation_space["screen"].shape
    gray_space = Box(low=0, high=255, shape=(h, w), dtype=np.uint8)
    env = TransformObservation(
        env,
        lambda obs: (obs["screen"] @ np.array([0.299, 0.587, 0.114])).astype(np.uint8),
        gray_space,
    )
    env = ResizeObservation(env, OBS_SIZE)
    env = FrameStackObservation(env, stack_size=STACK_SIZE)
    env = TransformReward(env, lambda r: r / 100.0)
    return env


def make_env(*, custom_scenario=False, doom_game_path=None):
    kwargs = {
        "render_mode": "rgb_array",
        "frame_skip": FRAME_SKIP,
        "screen_resolution": vzd.ScreenResolution.RES_200X125,
        "max_buttons_pressed": MAX_BUTTONS_PRESSED,
        "treat_episode_timeout_as_truncation": False,
    }
    if doom_game_path is not None:
        kwargs["doom_game_path"] = doom_game_path
    if custom_scenario:
        env = VizdoomEnv(config_file=custom_scenario_location, **kwargs)
    else:
        env = gymnasium.make(ENV_NAME, **kwargs)
    return preprocess_env(env)


def make_train_env():
    return make_env(doom_game_path=DOOM_WAD)


env = gymnasium.vector.SyncVectorEnv([make_train_env] * NUM_ENVS, autoreset_mode=gymnasium.vector.AutoresetMode.SAME_STEP,)


# RainbowDQN Hyperparameters

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
INIT_HP = {
    "BATCH_SIZE": 64,
    "LR": 1e-4,
    "GAMMA": 0.99,
    "MEMORY_SIZE": 100_000,
    "LEARN_STEP": 4,
    "N_STEP": 3,
    "ALPHA": 0.6,
    "BETA": 0.4,
    "TAU": 0.001,
    "NUM_ATOMS": 51,
    "V_MIN": -8.0,
    "V_MAX": 2.0,
    "LEARNING_DELAY": 1000,

    # Experiment schedule
    "MAX_STEPS": 50_000,

    # Eval every checkpoint
    "EVO_STEPS": 2_000,

    # Let each evaluation episode run normally
    "EVAL_STEPS": None,

    # 3 episodes per evaluation
    "EVAL_LOOP": 3,

    # 10 checkpoints total
    "CHECKPOINT": 2_000,
}
net_config = {
    "encoder_config": {
        "channel_size": [32, 64, 64],
        "kernel_size": [8, 4, 3],
        "stride_size": [4, 2, 1],
    },
    "head_config": {"hidden_size": [256], "max_mlp_nodes": 1024},
}


# Replay Buffers

In [ ]:
from agilerl.components.replay_buffer import ReplayBuffer

memory = ReplayBuffer(
    max_size=INIT_HP["MEMORY_SIZE"],
    device=device,
)

# Create RainbowDQN Agent

In [ ]:
from agilerl.algorithms.dqn import DQN

dqn_agent = DQN(
    observation_space=env.single_observation_space,
    action_space=env.single_action_space,
    net_config=net_config,
    batch_size=INIT_HP["BATCH_SIZE"],
    lr=INIT_HP["LR"],
    learn_step=INIT_HP["LEARN_STEP"],
    gamma=INIT_HP["GAMMA"],
    tau=INIT_HP["TAU"],
    device=device,
)

# Train with `train_off_policy()`

In [ ]:
trained_pop, pop_fitnesses = train_off_policy(
    env=env,
    env_name=ENV_NAME,
    algo="DQN",
    pop=[dqn_agent],
    memory=memory,
    init_hp=INIT_HP,

    max_steps=INIT_HP["MAX_STEPS"],
    evo_steps=INIT_HP["EVO_STEPS"],
    eval_steps=INIT_HP["EVAL_STEPS"],
    eval_loop=INIT_HP["EVAL_LOOP"],
    learning_delay=INIT_HP["LEARNING_DELAY"],

    tournament=None,
    mutation=None,
    wb=False,

    checkpoint=INIT_HP["CHECKPOINT"],
    checkpoint_path=os.path.join(checkpoint_dir, "DQN.pt"),
    overwrite_checkpoints=False,
        eps_start=1.0,
    eps_end=0.05,
    eps_decay=0.9995,
)

env.close()

Training Progress │   4% │ ▊                    │ 2000/50000 steps │ ⏱️ 00:12 │ ⏳ 05:08 │ 155.82step/s

    Global Steps: 2_000     
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric        ┃  Agent 0 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ fitness       │  -0.2167 │
├───────────────┼──────────┤
│ score         │  -1.6605 │
│ loss          │   0.0107 │
├───────────────┼──────────┤
│ lr            │ 1.00e-04 │
│ batch_size    │       64 │
│ learn_step    │        4 │
├───────────────┼──────────┤
│ steps         │     2000 │
│ last_mutation │     None │
│ steps/s       │ 155.8356 │
└───────────────┴──────────┘

Training Progress │   4% │ ▊                    │ 2000/50000 steps │ ⏱️ 00:13 │ ⏳ 05:08 │ 155.82step/s

    Global Steps: 2_000     
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric        ┃  Agent 0 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ fitness       │  -0.2167 │
├───────────────┼──────────┤
│ score         │  -1.6605 │
│ loss          │   0.0107 │
├───────────────┼──────────┤
│ lr            │ 1.00e-04 │
│ batch_size    │       64 │
│ learn_step    │        4 │
├───────────────┼──────────┤
│ steps         │     2000 │
│ last_mutation │     None │
│ steps/s       │ 155.8356 │
└───────────────┴──────────┘



Training Progress │   8% │ █▌                   │ 4000/50000 steps │ ⏱️ 00:22 │ ⏳ 04:07 │ 185.73step/s

...

    Global Steps: 48_000    
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric        ┃  Agent 0 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ fitness       │   0.5933 │
├───────────────┼──────────┤
│ score         │   0.6634 │
│ loss          │ 7.06e-03 │
├───────────────┼──────────┤
│ lr            │ 1.00e-04 │
│ batch_size    │       64 │
│ learn_step    │        4 │
├───────────────┼──────────┤
│ steps         │    48000 │
│ last_mutation │     None │
│ steps/s       │ 192.8387 │
└───────────────┴──────────┘



Training Progress │ 100% │ ████████████████████ │ 50000/50000 steps │ ⏱️ 04:11 │ ⏳ 00:00 │ 195.58step/s

    Global Steps: 50_000    
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric        ┃  Agent 0 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ fitness       │   0.7983 │
├───────────────┼──────────┤
│ score         │   0.6832 │
│ loss          │ 7.01e-03 │
├───────────────┼──────────┤
│ lr            │ 1.00e-04 │
│ batch_size    │       64 │
│ learn_step    │        4 │
├───────────────┼──────────┤
│ steps         │    50000 │
│ last_mutation │     None │
│ steps/s       │ 208.4336 │
└───────────────┴──────────┘

Training Progress │ 100% │ ████████████████████ │ 50000/50000 steps │ ⏱️ 04:11 │ ⏳ 00:00 │ 198.90step/s

    Global Steps: 50_000    
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric        ┃  Agent 0 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ fitness       │   0.7983 │
├───────────────┼──────────┤
│ score         │   0.6832 │
│ loss          │ 7.01e-03 │
├───────────────┼──────────┤
│ lr            │ 1.00e-04 │
│ batch_size    │       64 │
│ learn_step    │        4 │
├───────────────┼──────────┤
│ steps         │    50000 │
│ last_mutation │     None │
│ steps/s       │ 208.4336 │
└───────────────┴──────────┘



# Discover Saved Checkpoints

In [ ]:
def checkpoint_step(path):
    nums = [int(n) for n in re.findall(r"\d+", Path(path).stem)]
    return nums[-1] if nums else -1


checkpoints = sorted(Path(checkpoint_dir).glob("*.pt"), key=checkpoint_step)
assert checkpoints, f"No checkpoints found in {checkpoint_dir}"
checkpoints


# Create Evaluation Environment

In [ ]:
def reset_with_lmp(env, lmp_path, seed=None):
    base = env.unwrapped
    orig_reset = base.reset

    def recorded_reset(*, seed=None, options=None):
        gymnasium.Env.reset(base, seed=seed)
        game_seed = int(
            base.np_random.integers(0, np.iinfo(np.uint32).max + 1, dtype=np.uint32)
        )
        base.game.set_seed(game_seed)
        if not base.game.is_running():
            base.game.init()
        base.game.new_episode(str(lmp_path))
        base.state = base.game.get_state()
        return base._VizdoomEnv__collect_observations(), {}

    base.reset = recorded_reset
    try:
        return env.reset(seed=seed)
    finally:
        base.reset = orig_reset


def flush_lmp(env):
    env.unwrapped.game.new_episode()


eval_env = make_env(custom_scenario=use_custom_scenario, doom_game_path=DOOM_WAD)
replay_scenario = custom_scenario_location if use_custom_scenario else os.path.join(vzd.scenarios_path, "basic.cfg")


# Evaluate + Record Each Checkpoint

In [ ]:
for gen, ckpt in enumerate(checkpoints):
    agent = DQN.load(str(ckpt), device=device)
    agent.set_training_mode(False)
    print("----Generation %s----"%gen)
    generational_reward_average = 0
    for episode in range(EVAL_EPISODES):
        recording_file = os.path.join(recording_dir, f"gen_{gen}_episode_{episode}.lmp")
        obs, info = reset_with_lmp(eval_env, recording_file)
        done = False
        total_reward = 0
        while not done:
            action = agent.get_action(np.expand_dims(obs, 0), training=False)
            action = int(np.asarray(action).reshape(-1)[0])
            obs, reward, terminated, truncated, info = eval_env.step(action)
            done = terminated or truncated
            total_reward += reward
        print("--Episode %s reward=%s"%(episode, total_reward))
        generational_reward_average += total_reward
    generational_reward_average /= EVAL_EPISODES
    print("Generation Reward Average=%s"%generational_reward_average)

flush_lmp(eval_env)
eval_env.close()


----Generation 0----
--Episode 0 reward=-0.1299999999999999
--Episode 1 reward=-4.75
--Episode 2 reward=-4.7
--Episode 3 reward=-0.4600000000000001
--Episode 4 reward=-0.4600000000000001
Generation Reward Average=-2.1000000000000005
----Generation 1----
--Episode 0 reward=-3.500000000000002
--Episode 1 reward=-3.500000000000002
--Episode 2 reward=-3.500000000000002
--Episode 3 reward=-3.500000000000002
--Episode 4 reward=-3.500000000000002
Generation Reward Average=-3.500000000000002
----Generation 2----
--Episode 0 reward=0.2999999999999998
--Episode 1 reward=-3.500000000000002
--Episode 2 reward=-3.500000000000002
--Episode 3 reward=-3.500000000000002
--Episode 4 reward=-3.500000000000002
Generation Reward Average=-2.7400000000000015
----Generation 3----
--Episode 0 reward=-3.750000000000002
--Episode 1 reward=-0.07999999999999989
--Episode 2 reward=-0.05999999999999997
--Episode 3 reward=-3.750000000000002
--Episode 4 reward=-0.02999999999999995
Generation Reward Average=-1.53400000

# Replay / Convert `.lmp` Recordings

In [ ]:
SAMPLE_RATE = 22050


def lmp_to_mp4(
    lmp_path,
    mp4_path=None,
    *,
    doom_game_path=None,
    scenario=replay_scenario,
    fps=35,
):
    lmp_path = Path(lmp_path)
    mp4_path = Path(mp4_path) if mp4_path else lmp_path.with_suffix(".mp4")
    silent_path = mp4_path.with_name(mp4_path.stem + "_silent.mp4")
    wav_path = mp4_path.with_suffix(".wav")

    game = vzd.DoomGame()
    game.load_config(str(scenario))
    if doom_game_path is not None:
        game.set_doom_game_path(str(doom_game_path))
    #game.set_screen_resolution(vzd.ScreenResolution.RES_640X480)
    game.set_screen_resolution(vzd.ScreenResolution.RES_1920X1080)
    game.set_render_hud(True)
    game.set_render_corpses(True)
    game.set_window_visible(False)
    game.set_audio_buffer_enabled(True)
    game.set_audio_sampling_rate(vzd.SamplingRate.SR_22050)
    game.set_audio_buffer_size(1)
    game.add_game_args("+snd_efx 0")
    game.set_mode(vzd.Mode.SPECTATOR)
    game.init()

    frames, audio_slices = [], []
    try:
        game.replay_episode(str(lmp_path))
        while not game.is_episode_finished():
            state = game.get_state()
            if state is not None:
                if state.screen_buffer is not None:
                    frames.append(np.transpose(state.screen_buffer, (1, 2, 0)))
                if state.audio_buffer is not None:
                    audio_slices.append(state.audio_buffer)
            game.advance_action()
    finally:
        game.close()

    if not frames:
        raise RuntimeError(f"No frames decoded from {lmp_path}")

    imageio.mimsave(silent_path, frames, fps=fps, codec="libx264", quality=8)

    if audio_slices:
        audio = np.concatenate(audio_slices, axis=0)
        if audio.max() == 0:
            silent_path.replace(mp4_path)
        else:
            wavfile.write(str(wav_path), SAMPLE_RATE, audio)
            ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
            subprocess.check_call(
                [
                    ffmpeg,
                    "-y",
                    "-i",
                    str(silent_path),
                    "-i",
                    str(wav_path),
                    "-c:v",
                    "copy",
                    "-c:a",
                    "aac",
                    "-shortest",
                    str(mp4_path),
                ],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            silent_path.unlink(missing_ok=True)
            wav_path.unlink(missing_ok=True)
    else:
        silent_path.replace(mp4_path)

    return mp4_path




In [ ]:
def lmp_sort_key(path):
    match = re.search(r"gen_(\d+)_episode_(\d+)", path.name)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return (path.name,)

lmp_files = sorted(Path(recording_dir).glob("gen_*_episode_*.lmp"), key=lmp_sort_key)
lmp_files

In [ ]:


record_videos = True
if record_videos:
    mp4s = []
    for lmp in lmp_files:
        out = lmp_to_mp4(lmp, video_dir / f"{lmp.stem}.mp4", doom_game_path=DOOM_WAD)
        mp4s.append(out)
    mp4s


# Preview Results

In [ ]:
if record_videos:
    for mp4 in mp4s[-1:]:
        display(Video(str(mp4), embed=True, width=480))


Output hidden; open in https://colab.research.google.com to view.